# Phase 23 — Model API Contract and Integration Validation

This notebook proves notebook-exported model-core outputs can be safely consumed by the backend/API wrapper contract. It defines model-core request/response schemas, creates positive and negative contract fixtures from backend OpenAPI/docs, validates hard constraints, exercises safe fallback cases, and writes an integration readiness report that keeps backend-owned response fields outside training artifacts.


## Step 23.1 — Model-core request/response schemas

### Purpose
Define model-core request/response schemas for CV analysis and candidate reranking.

### Required input
Phase 22 exported model-core schema, backend OpenAPI contract, and backend CV Analyzer module docs.

### Action
Load source contracts and define strict model-core schemas for CV analysis and candidate reranking. Keep wrapper-owned fields out of model-core responses.

### Expected output
`model_core_contract` with request schemas, response schemas, forbidden fields, and mapping notes.

### Verification
Schemas include all model-owned outputs and exclude backend/wrapper-owned public response fields.


In [11]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

ROOT = Path.cwd()
if not (ROOT / 'GAP_MODEL_TRAINING.md').exists():
    ROOT = Path.cwd().parent.parent
REPORTS = ROOT / 'reports'
ARTIFACT_DIR = ROOT / 'artifacts/phase_23_model_api_contract_validation'
EXPORT_DIR = ROOT / 'artifacts/phase_22_calibration_model_card_export'
OPENAPI_PATH = ROOT / 'references/docs/generated/openapi.json'
CV_ANALYZER_DOC = ROOT / 'references/docs/modules/ai-cv-analyzer.md'
REPORTS.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PHASE_ID = 'phase_23_model_api_contract_validation'
SCHEMA_VERSION = 'model-api-contract-validation-v1'
GENERATED_AT = datetime.now(timezone.utc).isoformat()
MAX_JOB_ROLES = 10
MAX_RECOMMENDATIONS = 5
MAX_CANDIDATES = 25
ALLOWED_LANGUAGES = {'ID', 'EN'}
OPENAPI_LANGUAGES = {'id', 'en'}
WRAPPER_ONLY_FIELDS = {
    'topActionables', 'sectionReviews', 'title', 'companyName', 'reason',
    'nextStep', 'generatedCv', 'hydratedJob', 'visibility', 'availability',
}

openapi = json.loads(OPENAPI_PATH.read_text())
cv_doc_text = CV_ANALYZER_DOC.read_text()
phase22_schema = json.loads((EXPORT_DIR / 'model_core_schema.json').read_text())

model_core_contract: dict[str, Any] = {
    'schemaVersion': SCHEMA_VERSION,
    'sources': {
        'openapi': str(OPENAPI_PATH.relative_to(ROOT)),
        'cvAnalyzerDoc': str(CV_ANALYZER_DOC.relative_to(ROOT)),
        'phase22ModelCoreSchema': str((EXPORT_DIR / 'model_core_schema.json').relative_to(ROOT)),
    },
    'cvAnalysisRequest': {
        'required': ['requestId', 'inputVersion', 'language', 'jobRoles', 'cv'],
        'language': sorted(ALLOWED_LANGUAGES),
        'maxJobRoles': MAX_JOB_ROLES,
        'cv': ['fileId', 'mimeType', 'sizeBytes', 'parser'],
    },
    'cvAnalysisResponse': {
        'required': ['requestId', 'schemaVersion', 'jobFitAlignment', 'atsFriendliness', 'overallImpression', 'model', 'analyzedAt'],
        'schemaVersion': 'model-core-cv-analysis-v1',
        'scoreFields': ['jobFitAlignment.score', 'atsFriendliness.score', 'overallImpression.score'],
        'forbiddenFields': sorted(WRAPPER_ONLY_FIELDS),
    },
    'candidateRerankingRequest': {
        'required': ['requestId', 'candidateSetId', 'language', 'profile', 'jobCandidates'],
        'maxCandidates': MAX_CANDIDATES,
        'candidateRequired': ['jobId', 'roleTitle', 'requirements', 'skills'],
    },
    'candidateRerankingResponse': {
        'required': ['requestId', 'schemaVersion', 'recommendations', 'model', 'rankedAt'],
        'schemaVersion': 'model-core-candidate-reranking-v1',
        'maxRecommendations': MAX_RECOMMENDATIONS,
        'forbiddenFields': sorted(WRAPPER_ONLY_FIELDS),
    },
    'mappingPolicy': {
        'backendSchemaVersion': 'cv-analysis-v2',
        'languageMapping': {'ID': 'id', 'EN': 'en'},
        'backendOwned': sorted(WRAPPER_ONLY_FIELDS),
        'modelOwned': sorted(phase22_schema['outputs'].keys()),
    },
}

schema_checks = {
    'has_openapi_cv_analysis_v2': 'cv-analysis-v2' in json.dumps(openapi),
    'doc_mentions_model_api': 'Model API' in cv_doc_text,
    'phase22_forbidden_fields_loaded': set(phase22_schema['forbidden_model_core_fields']).issubset(WRAPPER_ONLY_FIELDS),
    'wrapper_fields_excluded_from_cv_response': not (set(model_core_contract['cvAnalysisResponse']['required']) & WRAPPER_ONLY_FIELDS),
    'wrapper_fields_excluded_from_reranking_response': not (set(model_core_contract['candidateRerankingResponse']['required']) & WRAPPER_ONLY_FIELDS),
}
assert all(schema_checks.values()), schema_checks


## Step 23.2 — Contract fixtures aligned with backend OpenAPI and module docs

### Purpose
Add notebook contract fixtures aligned with `references/docs/generated/openapi.json` and `references/docs/modules/ai-cv-analyzer.md`.

### Required input
OpenAPI CV Analyzer request/response schemas, module documentation, and model-core schema policy.

### Action
Build positive fixtures for CV analysis and candidate reranking plus negative fixtures for invalid outputs and invalid candidate sets.

### Expected output
`contract_fixtures` covering valid and invalid integration examples.

### Verification
Fixtures include backend-compatible language mapping, `cv-analysis-v2` mapping context, positive cases, and negative cases.


In [12]:
cv_positive_request = {
    'requestId': 'req_phase23_cv_ok',
    'inputVersion': 'cv-analyzer-v1',
    'language': 'ID',
    'inputMode': 'UPLOAD',
    'compareSource': 'JOB_SEARCH',
    'cv': {
        'fileId': 'cv_file_phase23',
        'mimeType': 'application/pdf',
        'sizeBytes': 524288,
        'parser': {'status': 'parsed', 'textLength': 4200, 'confidence': 0.91},
    },
    'jobRoles': ['Backend Developer', 'Software Engineer'],
}

cv_positive_response = {
    'requestId': 'req_phase23_cv_ok',
    'schemaVersion': 'model-core-cv-analysis-v1',
    'jobFitAlignment': {
        'score': 78,
        'summarySignals': [{'key': 'skill_overlap', 'label': 'REST API and PostgreSQL evidence found'}],
        'matchedSkills': ['rest api', 'postgresql', 'typescript'],
        'missingSkills': ['deployment evidence'],
        'confidenceNotes': ['E5-backed job-fit evidence available'],
    },
    'atsFriendliness': {
        'score': 84,
        'detectedIssues': ['skills section could be clearer'],
        'evidence': {'sectionCompleteness': 'present', 'parseableText': True},
        'fallback': False,
    },
    'overallImpression': {
        'score': 80,
        'summary': 'CV shows backend evidence with clear REST API and PostgreSQL signals; deployment evidence is still sparse.',
        'evidenceKeys': ['skill_overlap', 'ats_parseable_text'],
        'confidenceNotes': ['No unsupported seniority or hiring-outcome claim detected'],
    },
    'model': {'name': 'bisakerja-model-core', 'version': 'phase22-export'},
    'analyzedAt': GENERATED_AT,
}

reranking_positive_request = {
    'requestId': 'req_phase23_rank_ok',
    'candidateSetId': 'candidate_set_phase23',
    'language': 'EN',
    'profile': {'skills': ['typescript', 'postgresql', 'rest api'], 'role': 'Backend Developer'},
    'jobCandidates': [
        {'jobId': 'job_backend_1', 'roleTitle': 'Backend Developer', 'requirements': ['REST API', 'PostgreSQL'], 'skills': ['typescript', 'postgresql']},
        {'jobId': 'job_frontend_1', 'roleTitle': 'Frontend Developer', 'requirements': ['React', 'CSS'], 'skills': ['react', 'css']},
        {'jobId': 'job_fullstack_1', 'roleTitle': 'Fullstack Developer', 'requirements': ['TypeScript', 'PostgreSQL'], 'skills': ['typescript', 'postgresql', 'react']},
    ],
}

reranking_positive_response = {
    'requestId': 'req_phase23_rank_ok',
    'schemaVersion': 'model-core-candidate-reranking-v1',
    'recommendations': [
        {'jobId': 'job_backend_1', 'matchScore': 88, 'matchLevel': 'strong', 'matchedSkills': ['rest api', 'postgresql'], 'missingSkills': [], 'rankingSignals': [{'key': 'skill_overlap', 'value': 1.0}]},
        {'jobId': 'job_fullstack_1', 'matchScore': 76, 'matchLevel': 'good', 'matchedSkills': ['typescript', 'postgresql'], 'missingSkills': ['react'], 'rankingSignals': [{'key': 'skill_overlap', 'value': 0.67}]},
    ],
    'model': {'name': 'bisakerja-model-core', 'version': 'phase22-export'},
    'rankedAt': GENERATED_AT,
}

negative_fixtures = {
    'cv_score_out_of_range': {**cv_positive_response, 'jobFitAlignment': {**cv_positive_response['jobFitAlignment'], 'score': 101}},
    'cv_missing_model_metadata': {k: v for k, v in cv_positive_response.items() if k != 'model'},
    'cv_wrapper_owned_field': {**cv_positive_response, 'topActionables': ['Backend wrapper owns this field']},
    'cv_invalid_language': {**cv_positive_request, 'language': 'JP'},
    'rerank_unknown_job': {**reranking_positive_response, 'recommendations': [{**reranking_positive_response['recommendations'][0], 'jobId': 'job_not_in_candidate_set'}]},
    'rerank_duplicate_job': {**reranking_positive_response, 'recommendations': [reranking_positive_response['recommendations'][0], reranking_positive_response['recommendations'][0]]},
    'rerank_too_many_items': {**reranking_positive_response, 'recommendations': reranking_positive_response['recommendations'] * 3},
    'rerank_wrapper_owned_field': {**reranking_positive_response, 'recommendations': [{**reranking_positive_response['recommendations'][0], 'title': 'Backend Developer'}]},
}

contract_fixtures = {
    'positive': {
        'cvAnalysisRequest': cv_positive_request,
        'cvAnalysisResponse': cv_positive_response,
        'candidateRerankingRequest': reranking_positive_request,
        'candidateRerankingResponse': reranking_positive_response,
    },
    'negative': negative_fixtures,
    'backendMappingExample': {
        'targetSchemaVersion': 'cv-analysis-v2',
        'language': {'modelCore': 'ID', 'backend': 'id'},
        'jobFitAlignment': {'score': 78, 'summarySource': 'backend wrapper renders from summarySignals'},
        'atsFriendliness': {'score': 84, 'summarySource': 'backend wrapper renders from detectedIssues'},
        'overallImpression': 'backend wrapper may render product-safe string from model-core summary',
    },
}

fixture_checks = {
    'positive_cv_language_maps_to_openapi': model_core_contract['mappingPolicy']['languageMapping'][cv_positive_request['language']] in OPENAPI_LANGUAGES,
    'positive_cv_has_required_outputs': set(model_core_contract['cvAnalysisResponse']['required']).issubset(cv_positive_response),
    'positive_rerank_has_required_outputs': set(model_core_contract['candidateRerankingResponse']['required']).issubset(reranking_positive_response),
    'negative_fixtures_present': len(negative_fixtures) >= 8,
}
assert all(fixture_checks.values()), fixture_checks


## Step 23.3 — Hard contract validation checks

### Purpose
Validate score bounds, required fields, language handling, candidate membership, duplicate rejection, max item counts, and model metadata.

### Required input
Model-core schemas and contract fixtures.

### Action
Run positive validators and expected negative rejections for CV analysis and candidate reranking.

### Expected output
`validation_results` with zero positive violations and expected negative rejections.

### Verification
Every required hard check is represented and all negative fixtures fail for the intended reason.


In [13]:
def has_forbidden_field(value: Any, path: str = '$') -> list[str]:
    hits: list[str] = []
    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f'{path}.{key}'
            if key in WRAPPER_ONLY_FIELDS:
                hits.append(child_path)
            hits.extend(has_forbidden_field(child, child_path))
    elif isinstance(value, list):
        for idx, child in enumerate(value):
            hits.extend(has_forbidden_field(child, f'{path}[{idx}]'))
    return hits


def score_errors(output: dict[str, Any]) -> list[str]:
    paths = [
        ('jobFitAlignment.score', output.get('jobFitAlignment', {}).get('score')),
        ('atsFriendliness.score', output.get('atsFriendliness', {}).get('score')),
        ('overallImpression.score', output.get('overallImpression', {}).get('score')),
    ]
    errors = []
    for path, score in paths:
        if not isinstance(score, int) or not 0 <= score <= 100:
            errors.append(path)
    return errors


def validate_cv_request(req: dict[str, Any]) -> list[dict[str, Any]]:
    errors = []
    for field in model_core_contract['cvAnalysisRequest']['required']:
        if field not in req:
            errors.append({'check': 'required_request_field', 'field': field})
    if req.get('language') not in ALLOWED_LANGUAGES:
        errors.append({'check': 'language_enum', 'value': req.get('language')})
    if len(req.get('jobRoles', [])) > MAX_JOB_ROLES or not req.get('jobRoles'):
        errors.append({'check': 'job_roles_count', 'count': len(req.get('jobRoles', []))})
    if req.get('cv', {}).get('mimeType') != 'application/pdf':
        errors.append({'check': 'cv_mime_type', 'value': req.get('cv', {}).get('mimeType')})
    return errors


def validate_cv_response(output: dict[str, Any]) -> list[dict[str, Any]]:
    errors = []
    for field in model_core_contract['cvAnalysisResponse']['required']:
        if field not in output:
            errors.append({'check': 'required_response_field', 'field': field})
    if output.get('schemaVersion') != model_core_contract['cvAnalysisResponse']['schemaVersion']:
        errors.append({'check': 'schema_version', 'value': output.get('schemaVersion')})
    for path in score_errors(output):
        errors.append({'check': 'score_bounds', 'field': path})
    forbidden = has_forbidden_field(output)
    if forbidden:
        errors.append({'check': 'wrapper_owned_field', 'paths': forbidden})
    model = output.get('model')
    if not isinstance(model, dict) or not model.get('name') or not model.get('version'):
        errors.append({'check': 'model_metadata', 'value': model})
    return errors


def validate_rerank_request(req: dict[str, Any]) -> list[dict[str, Any]]:
    errors = []
    for field in model_core_contract['candidateRerankingRequest']['required']:
        if field not in req:
            errors.append({'check': 'required_request_field', 'field': field})
    candidates = req.get('jobCandidates', [])
    if req.get('language') not in ALLOWED_LANGUAGES:
        errors.append({'check': 'language_enum', 'value': req.get('language')})
    if not candidates or len(candidates) > MAX_CANDIDATES:
        errors.append({'check': 'candidate_count', 'count': len(candidates)})
    ids = [row.get('jobId') for row in candidates]
    if len(ids) != len(set(ids)):
        errors.append({'check': 'duplicate_candidate_input', 'ids': ids})
    return errors


def validate_rerank_response(req: dict[str, Any], output: dict[str, Any]) -> list[dict[str, Any]]:
    errors = []
    for field in model_core_contract['candidateRerankingResponse']['required']:
        if field not in output:
            errors.append({'check': 'required_response_field', 'field': field})
    if output.get('schemaVersion') != model_core_contract['candidateRerankingResponse']['schemaVersion']:
        errors.append({'check': 'schema_version', 'value': output.get('schemaVersion')})
    rows = output.get('recommendations', [])
    candidate_ids = {row.get('jobId') for row in req.get('jobCandidates', [])}
    ids = [row.get('jobId') for row in rows]
    unknown = sorted(set(ids) - candidate_ids)
    if unknown:
        errors.append({'check': 'candidate_membership', 'unknownIds': unknown})
    duplicates = sorted({job_id for job_id in ids if ids.count(job_id) > 1})
    if duplicates:
        errors.append({'check': 'duplicate_recommendations', 'jobIds': duplicates})
    if len(rows) > MAX_RECOMMENDATIONS:
        errors.append({'check': 'max_item_count', 'count': len(rows)})
    for idx, row in enumerate(rows):
        score = row.get('matchScore')
        if not isinstance(score, int) or not 0 <= score <= 100:
            errors.append({'check': 'score_bounds', 'field': f'recommendations[{idx}].matchScore'})
    forbidden = has_forbidden_field(output)
    if forbidden:
        errors.append({'check': 'wrapper_owned_field', 'paths': forbidden})
    model = output.get('model')
    if not isinstance(model, dict) or not model.get('name') or not model.get('version'):
        errors.append({'check': 'model_metadata', 'value': model})
    return errors

positive_validation = {
    'cvRequest': validate_cv_request(cv_positive_request),
    'cvResponse': validate_cv_response(cv_positive_response),
    'rerankingRequest': validate_rerank_request(reranking_positive_request),
    'rerankingResponse': validate_rerank_response(reranking_positive_request, reranking_positive_response),
}

negative_validation = {
    name: (
        validate_cv_request(fixture) if name == 'cv_invalid_language'
        else validate_rerank_response(reranking_positive_request, fixture) if name.startswith('rerank_')
        else validate_cv_response(fixture)
    )
    for name, fixture in negative_fixtures.items()
}

validation_results = {
    'positive': positive_validation,
    'negative': negative_validation,
    'summary': {
        'positiveViolationCount': sum(len(v) for v in positive_validation.values()),
        'negativeRejectedCount': sum(1 for v in negative_validation.values() if v),
        'negativeFixtureCount': len(negative_validation),
        'coveredChecks': sorted({err['check'] for errs in negative_validation.values() for err in errs}),
    },
}

required_negative_checks = {'score_bounds', 'model_metadata', 'wrapper_owned_field', 'language_enum', 'candidate_membership', 'duplicate_recommendations', 'max_item_count'}
validation_checks = {
    'positive_has_zero_violations': validation_results['summary']['positiveViolationCount'] == 0,
    'all_negative_fixtures_rejected': validation_results['summary']['negativeRejectedCount'] == validation_results['summary']['negativeFixtureCount'],
    'required_checks_covered': required_negative_checks.issubset(set(validation_results['summary']['coveredChecks'])),
}
assert all(validation_checks.values()), validation_checks


## Step 23.4 — Safe fallback validation

### Purpose
Validate safe fallbacks for timeout, model unavailable, invalid model output, empty CV parse, and low confidence.

### Required input
Failure-mode policy, parser confidence fixtures, and hard validators.

### Action
Create fallback scenarios and verify each maps to a safe backend action without exposing raw internals or persisting invalid model output.

### Expected output
`fallback_validation` with pass/fail evidence for every required failure mode.

### Verification
Timeout, unavailable, invalid output, empty CV parse, and low-confidence cases are all covered.


In [14]:
fallback_scenarios = [
    {
        'name': 'timeout',
        'input': {'error': 'MODEL_TIMEOUT', 'requestId': 'req_timeout'},
        'expectedAction': 'backend_returns_retryable_error_or_safe_processing_message',
        'persistModelOutput': False,
        'exposeRawInternals': False,
    },
    {
        'name': 'model_unavailable',
        'input': {'error': 'MODEL_UNAVAILABLE', 'requestId': 'req_unavailable'},
        'expectedAction': 'backend_returns_service_unavailable_without_partial_scores',
        'persistModelOutput': False,
        'exposeRawInternals': False,
    },
    {
        'name': 'invalid_model_output',
        'input': negative_fixtures['cv_score_out_of_range'],
        'expectedAction': 'reject_before_persistence_or_user_response',
        'persistModelOutput': False,
        'exposeRawInternals': False,
    },
    {
        'name': 'empty_cv_parse',
        'input': {**cv_positive_request, 'cv': {**cv_positive_request['cv'], 'parser': {'status': 'empty', 'textLength': 0, 'confidence': 0.0}}},
        'expectedAction': 'return_safe_empty_parse_fallback_with_no_skill_claims',
        'persistModelOutput': True,
        'exposeRawInternals': False,
    },
    {
        'name': 'low_confidence',
        'input': {**cv_positive_response, 'overallImpression': {**cv_positive_response['overallImpression'], 'score': 45, 'confidenceNotes': ['low confidence: sparse evidence']}, 'jobFitAlignment': {**cv_positive_response['jobFitAlignment'], 'confidenceNotes': ['low confidence: sparse evidence']}},
        'expectedAction': 'allow_bounded_low_confidence_output_with_visible_confidence_note',
        'persistModelOutput': True,
        'exposeRawInternals': False,
    },
]


def validate_fallback(scenario: dict[str, Any]) -> dict[str, Any]:
    name = scenario['name']
    checks = {
        'has_expected_action': bool(scenario.get('expectedAction')),
        'does_not_expose_raw_internals': scenario.get('exposeRawInternals') is False,
    }
    if name == 'invalid_model_output':
        checks['hard_validator_rejects'] = bool(validate_cv_response(scenario['input']))
        checks['not_persisted'] = scenario.get('persistModelOutput') is False
    if name in {'timeout', 'model_unavailable'}:
        checks['not_persisted'] = scenario.get('persistModelOutput') is False
    if name == 'empty_cv_parse':
        parser = scenario['input']['cv']['parser']
        checks['empty_parse_detected'] = parser['textLength'] == 0 and parser['status'] == 'empty'
        checks['safe_to_persist_fallback'] = scenario.get('persistModelOutput') is True
    if name == 'low_confidence':
        notes = scenario['input']['overallImpression']['confidenceNotes'] + scenario['input']['jobFitAlignment']['confidenceNotes']
        checks['confidence_note_present'] = any('low confidence' in note for note in notes)
        checks['safe_to_persist_bounded_output'] = scenario.get('persistModelOutput') is True
    return {'name': name, 'checks': checks, 'passed': all(checks.values()), 'expectedAction': scenario['expectedAction']}

fallback_validation = [validate_fallback(s) for s in fallback_scenarios]
fallback_checks = {
    'all_required_fallbacks_present': {'timeout', 'model_unavailable', 'invalid_model_output', 'empty_cv_parse', 'low_confidence'} == {row['name'] for row in fallback_validation},
    'all_fallbacks_pass': all(row['passed'] for row in fallback_validation),
}
assert all(fallback_checks.values()), fallback_checks


## Step 23.5 — Integration readiness report

### Purpose
Produce an integration readiness report that separates model-core outputs from backend/wrapper-owned response fields.

### Required input
Schemas, fixtures, validation results, fallback checks, and acceptance criteria.

### Action
Write contract fixtures and readiness report to disk. Summarize acceptance criteria and remaining integration notes.

### Expected output
`reports/phase_23_model_api_contract_validation.json` plus fixture artifacts under `artifacts/phase_23_model_api_contract_validation/`.

### Verification
Acceptance criteria pass, positive and negative fixtures are included, invalid output is rejected, and backend-only fields remain out of model-core artifacts.


In [15]:
acceptance = {
    'model_core_output_maps_to_cv_analysis_v2_without_raw_internals': (
        schema_checks['has_openapi_cv_analysis_v2']
        and fixture_checks['positive_cv_language_maps_to_openapi']
        and not has_forbidden_field(cv_positive_response)
    ),
    'invalid_model_output_rejected_before_persistence_or_user_response': bool(negative_validation['cv_score_out_of_range']),
    'notebook_contract_checks_include_positive_and_negative_fixtures': (
        validation_results['summary']['positiveViolationCount'] == 0
        and validation_results['summary']['negativeRejectedCount'] == validation_results['summary']['negativeFixtureCount']
    ),
    'backend_wrapper_only_fields_remain_outside_training_artifacts': (
        not has_forbidden_field(cv_positive_response)
        and not has_forbidden_field(reranking_positive_response)
        and set(model_core_contract['mappingPolicy']['backendOwned']) == WRAPPER_ONLY_FIELDS
    ),
}

readiness_status = 'ready_for_backend_integration_fixture_review' if all(acceptance.values()) and all(row['passed'] for row in fallback_validation) else 'blocked'

report = {
    'phase': 23,
    'phaseId': PHASE_ID,
    'generatedAt': GENERATED_AT,
    'schemaVersion': SCHEMA_VERSION,
    'readinessStatus': readiness_status,
    'sources': model_core_contract['sources'],
    'schemaChecks': schema_checks,
    'fixtureChecks': fixture_checks,
    'validationChecks': validation_checks,
    'fallbackChecks': fallback_checks,
    'validationResults': validation_results,
    'fallbackValidation': fallback_validation,
    'acceptance': acceptance,
    'modelCoreContract': model_core_contract,
    'integrationBoundary': {
        'modelCoreOwns': model_core_contract['mappingPolicy']['modelOwned'],
        'backendWrapperOwns': model_core_contract['mappingPolicy']['backendOwned'],
        'persistenceRule': 'persist only outputs that pass hard validators; never persist invalid model output',
        'rawInternalsRule': 'do not expose raw evidence internals directly in public response; backend wrapper renders product-safe copy',
    },
    'notes': [
        'Backend still owns public topActionables, sectionReviews, generatedCv, job hydration, reason, and nextStep copy.',
        'Model-core recommendations only rank backend-supplied candidate job IDs.',
        'Fallback cases define integration behavior; backend implementation must map them to actual HTTP errors/responses.',
    ],
}

(ARTIFACT_DIR / 'model_core_contract.json').write_text(json.dumps(model_core_contract, indent=2, sort_keys=True))
(ARTIFACT_DIR / 'contract_fixtures.json').write_text(json.dumps(contract_fixtures, indent=2, sort_keys=True))
(ARTIFACT_DIR / 'validation_results.json').write_text(json.dumps(validation_results, indent=2, sort_keys=True))
(ARTIFACT_DIR / 'fallback_validation.json').write_text(json.dumps(fallback_validation, indent=2, sort_keys=True))
(REPORTS / f'{PHASE_ID}.json').write_text(json.dumps(report, indent=2, sort_keys=True))

assert all(acceptance.values()), acceptance
assert readiness_status == 'ready_for_backend_integration_fixture_review', readiness_status
report


{'phase': 23,
 'phaseId': 'phase_23_model_api_contract_validation',
 'generatedAt': '2026-06-02T06:07:17.296442+00:00',
 'schemaVersion': 'model-api-contract-validation-v1',
 'readinessStatus': 'ready_for_backend_integration_fixture_review',
 'sources': {'openapi': 'references/docs/generated/openapi.json',
  'cvAnalyzerDoc': 'references/docs/modules/ai-cv-analyzer.md',
  'phase22ModelCoreSchema': 'artifacts/phase_22_calibration_model_card_export/model_core_schema.json'},
 'schemaChecks': {'has_openapi_cv_analysis_v2': True,
  'doc_mentions_model_api': True,
  'phase22_forbidden_fields_loaded': True,
  'wrapper_fields_excluded_from_cv_response': True,
  'wrapper_fields_excluded_from_reranking_response': True},
 'fixtureChecks': {'positive_cv_language_maps_to_openapi': True,
  'positive_cv_has_required_outputs': True,
  'positive_rerank_has_required_outputs': True,
  'negative_fixtures_present': True},
 'validationChecks': {'positive_has_zero_violations': True,
  'all_negative_fixtures_r